# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.9 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
torch.set_num_threads(1)

TASK_ID='task092'
CH=10
H=W=30
ROOT=Path(COMPETITION)
TASK_PATH=ROOT/f'{TASK_ID}.json'
if not TASK_PATH.exists():
    TASK_PATH=Path(LOCAL_DATA)/f'{TASK_ID}.json'
OUT_DIR=Path.cwd()
ONNX_PATH=OUT_DIR/f'{TASK_ID}_static_graph.onnx'
ZIP_PATH=OUT_DIR/f'{TASK_ID}_static_graph_submission.zip'
SUBMISSION_ZIP=OUT_DIR/f'{TASK_ID}_zero_pad_submission.zip'
GENERIC_ZIP=OUT_DIR/'submission.zip'
AUDIT_JSON=OUT_DIR/f'{TASK_ID}_zero_pad_audit.json'
AUDIT_CSV=OUT_DIR/f'{TASK_ID}_zero_pad_audit.csv'
print('TASK_PATH =', TASK_PATH)

TASK_PATH = /kaggle/input/competitions/neurogolf-2026/task092.json


In [6]:
def grid_to_tensor(grid):
    # Kaggle/reference contract: real cells are one-hot; padded cells are all-zero.
    # Do NOT fill padding with background channel 0.
    g=np.array(grid,dtype=np.int64)
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    h,w=g.shape
    if h>H or w>W:
        raise ValueError(f'grid too large for static shape: {g.shape}')
    for c in range(CH):
        x[0,c,:h,:w]=(g==c)
    return x

def expected_tensor(grid):
    return grid_to_tensor(grid)

def argmax_grid(y, shape):
    return y[0].argmax(axis=0)[:shape[0],:shape[1]]

def full_expected_onehot(grid):
    # Also zero outside the true output grid.
    return grid_to_tensor(grid)

def exact_eval(sess, examples):
    exact=0; total=0; first_wrong=None; raw_ok=True
    for i,ex in enumerate(examples):
        out=np.array(ex['output'],dtype=np.int64)
        if out.shape[0]>H or out.shape[1]>W:
            continue
        x=grid_to_tensor(ex['input'])
        y=sess.run(None, {'input':x})[0]
        pred=argmax_grid(y,out.shape)
        ok=bool(np.array_equal(pred,out))
        exp_full=full_expected_onehot(out)
        raw_binary=(y>0.5).astype(np.float32)
        raw_equal=bool(np.array_equal(raw_binary, exp_full))
        onehot_values=bool(np.all((np.abs(y)<1e-5) | (np.abs(y-1)<1e-5)))
        # Inside the true grid, channel sum is 1; outside it is 0. raw_equal enforces exact target.
        channel_sums_valid=bool(np.all((np.abs(y.sum(axis=1))<1e-5) | (np.abs(y.sum(axis=1)-1)<1e-5)))
        exact += int(ok and raw_equal and onehot_values and channel_sums_valid)
        total += 1
        if first_wrong is None and not (ok and raw_equal and onehot_values and channel_sums_valid):
            first_wrong={
                'index':i,
                'argmax_ok':ok,
                'raw_equal':raw_equal,
                'onehot_values':onehot_values,
                'channel_sums_valid':channel_sums_valid,
                'unique_values':[float(v) for v in np.unique(y)[:12]],
                'unique_channel_sums':[float(v) for v in np.unique(y.sum(axis=1))[:12]],
                'wrong_raw_pixels':int(np.abs(raw_binary-exp_full).sum()),
            }
    return {'exact':exact,'total':total,'first_wrong':first_wrong,'raw_ok':first_wrong is None}

def forbidden_report(model):
    bad={'Loop','Scan','NonZero','Unique','Script','Function','TreeEnsembleClassifier','TreeEnsembleRegressor'}
    risky={'Shape','Gather','ConstantOfShape','Expand','Range','ScatterND'}
    ops=collections.Counter(n.op_type for n in model.graph.node)
    return ops, sorted(set(ops)&bad), sorted(set(ops)&risky)

In [7]:
class Task092PairLineConnector(nn.Module):
    """Static, non-tree tensor program for task092.

    For each foreground color independently:
    - if two endpoints share a row, fill the closed horizontal interval between them;
    - if two endpoints share a column, fill the closed vertical interval between them;
    - if a horizontal and vertical segment cross, the vertical segment wins.

    This is implemented using fixed triangular prefix/suffix matrices, so there is no
    dynamic shape construction and no data-dependent Python control flow in ONNX.
    """
    def __init__(self, H=30, W=30):
        super().__init__()
        self.register_buffer('col_left_matrix', torch.triu(torch.ones(W,W,dtype=torch.float32)))
        self.register_buffer('col_right_matrix', torch.tril(torch.ones(W,W,dtype=torch.float32)))
        self.register_buffer('row_top_matrix', torch.tril(torch.ones(H,H,dtype=torch.float32)))
        self.register_buffer('row_bottom_matrix', torch.triu(torch.ones(H,H,dtype=torch.float32)))
        self.H=H; self.W=W

    def forward(self, x):
        m=x[:,1:10,:,:]  # foreground channels only: [1,9,30,30]

        # Rows/columns containing a pair of same-colored endpoints.
        hrow=(m.sum(dim=3,keepdim=True)>1.5).float()
        vcol=(m.sum(dim=2,keepdim=True)>1.5).float()

        # Horizontal closed interval: there is a marker on the left and right.
        left=torch.matmul(m,self.col_left_matrix)
        right=torch.matmul(m,self.col_right_matrix)
        hfill=((left>0.5).float()*(right>0.5).float()*hrow)

        # Vertical closed interval: there is a marker above and below.
        B,C,HH,WW=m.shape
        mm=m.reshape(B*C,HH,WW)
        top=torch.matmul(self.row_top_matrix.unsqueeze(0),mm).reshape(B,C,HH,WW)
        bottom=torch.matmul(self.row_bottom_matrix.unsqueeze(0),mm).reshape(B,C,HH,WW)
        vfill=((top>0.5).float()*(bottom>0.5).float()*vcol)

        # Vertical priority at intersections.
        v_union=torch.clamp(vfill.sum(dim=1,keepdim=True),0,1)
        fg=torch.clamp(vfill + hfill*(1-v_union),0,1)

        # Kaggle/reference contract: produce one-hot only inside the real grid.
        # Padded cells outside the grid must be all-zero, not background channel 0.
        active=(x.sum(dim=1,keepdim=True)>0.5).float()
        fg=fg*active
        bg=active*(1-torch.clamp(fg.sum(dim=1,keepdim=True),0,1))
        return torch.cat([bg,fg],dim=1)

In [8]:
with open(TASK_PATH) as f:
    task=json.load(f)
print({k:len(task.get(k,[])) for k in ['train','test','arc-gen']})
print('train shapes:', [np.array(e['input']).shape for e in task['train']])
print('test shapes:', [np.array(e['input']).shape for e in task['test']])

{'train': 2, 'test': 1, 'arc-gen': 262}
train shapes: [(30, 20), (20, 10)]
test shapes: [(20, 20)]


In [9]:
model=Task092PairLineConnector(H,W).eval()
dummy=torch.zeros(1,CH,H,W,dtype=torch.float32)

torch.onnx.export(model,dummy,str(ONNX_PATH),
                  input_names=['input'],output_names=['output'],
                  opset_version=13,dynamic_axes=None,
                  do_constant_folding=True,dynamo=False)

m=onnx.load(str(ONNX_PATH))
onnx.checker.check_model(m)
ops, forbidden, risky = forbidden_report(m)
health={
    'task_id':TASK_ID,
    'onnx_size_bytes':ONNX_PATH.stat().st_size,
    'input_shape':[d.dim_value for d in m.graph.input[0].type.tensor_type.shape.dim],
    'output_shape':[d.dim_value for d in m.graph.output[0].type.tensor_type.shape.dim],
    'ops':dict(sorted(ops.items())),
    'forbidden_ops':forbidden,
    'risky_dynamic_shape_ops':risky,
}
if health['onnx_size_bytes'] >= 1_400_000:
    raise AssertionError('ONNX too large')
if forbidden:
    raise AssertionError(f'Forbidden ops: {forbidden}')
if risky:
    raise AssertionError(f'Risky dynamic-shape ops: {risky}')
print(json.dumps(health,indent=2))

/tmp/ipykernel_16/1590855486.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(ONNX_PATH),


{
  "task_id": "task092",
  "onnx_size_bytes": 19788,
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "ops": {
    "Add": 1,
    "Cast": 7,
    "Clip": 3,
    "Concat": 1,
    "Constant": 25,
    "Greater": 7,
    "MatMul": 4,
    "Mul": 7,
    "ReduceSum": 5,
    "Reshape": 3,
    "Slice": 1,
    "Sub": 2
  },
  "forbidden_ops": [],
  "risky_dynamic_shape_ops": []
}


In [10]:
sess=ort.InferenceSession(str(ONNX_PATH),providers=['CPUExecutionProvider'])
for split in ['train','test']:
    res=exact_eval(sess,task.get(split,[]))
    health[f'{split}_exact']=f"{res['exact']}/{res['total']}"
    if res['first_wrong'] is not None:
        raise AssertionError((split,res))

arc=task.get('arc-gen',[])
arc_valid=[e for e in arc if np.array(e['output']).shape[0]<=H and np.array(e['output']).shape[1]<=W]
res_all=exact_eval(sess,arc_valid)
health['arc_gen_all']=f"{res_all['exact']}/{res_all['total']}"
if res_all['first_wrong'] is not None:
    raise AssertionError(('arc-gen all',res_all))

# 60% arc-gen holdout: final 60%, not random cherry-picking.
cut=int(len(arc_valid)*0.40)
holdout=arc_valid[cut:]
res_hold=exact_eval(sess,holdout)
health['arc_gen_holdout_60pct']=f"{res_hold['exact']}/{res_hold['total']}"
if res_hold['first_wrong'] is not None:
    raise AssertionError(('arc-gen holdout',res_hold))
print(json.dumps({k:health[k] for k in ['train_exact','test_exact','arc_gen_all','arc_gen_holdout_60pct']},indent=2))

{
  "train_exact": "2/2",
  "test_exact": "1/1",
  "arc_gen_all": "262/262",
  "arc_gen_holdout_60pct": "158/158"
}


In [11]:
# Synthetic OOD validation for the true rule, avoiding ambiguous multi-line overlaps.
def solve_pair_lines(inp):
    arr=np.array(inp,dtype=np.int64)
    out=arr.copy()
    h,w=arr.shape
    h_lines=[]; v_lines=[]
    for color in range(1,10):
        pts=list(zip(*np.where(arr==color)))
        if len(pts)<2:
            continue
        rows=[p[0] for p in pts]; cols=[p[1] for p in pts]
        if len(set(rows))==1:
            r=rows[0]; c1=min(cols); c2=max(cols)
            h_lines.append((color,r,c1,c2))
        if len(set(cols))==1:
            c=cols[0]; r1=min(rows); r2=max(rows)
            v_lines.append((color,c,r1,r2))
    for color,r,c1,c2 in h_lines:
        out[r,c1:c2+1]=color
    for color,c,r1,r2 in v_lines:
        out[r1:r2+1,c]=color
    return out

rng=random.Random(92092)
synthetic=[]
for _ in range(1200):
    h=rng.randint(6,30); w=rng.randint(6,30)
    arr=np.zeros((h,w),dtype=np.int64)
    colors=rng.sample(range(1,10), rng.randint(1,5))
    occupied_cells=set()
    occupied_endpoints=set()
    for color in colors:
        vertical = rng.random() < 0.55
        for attempt in range(100):
            if vertical:
                c=rng.randrange(w); r1,r2=sorted(rng.sample(range(h),2))
                if r2-r1<2: continue
                pts=[(r1,c),(r2,c)]
                segment={(r,c) for r in range(r1,r2+1)}
            else:
                r=rng.randrange(h); c1,c2=sorted(rng.sample(range(w),2))
                if c2-c1<2: continue
                pts=[(r,c1),(r,c2)]
                segment={(r,c) for c in range(c1,c2+1)}
            # Keep OOD unambiguous: no overlapping full segments and no repeated same-color endpoints.
            if segment.isdisjoint(occupied_cells) and all(p not in occupied_endpoints for p in pts):
                for p in pts:
                    occupied_endpoints.add(p); arr[p]=color
                occupied_cells |= segment
                break
    out=solve_pair_lines(arr)
    synthetic.append({'input':arr.tolist(),'output':out.tolist()})
res_syn=exact_eval(sess,synthetic)
health['synthetic_ood_nonoverlap']=f"{res_syn['exact']}/{res_syn['total']}"
if res_syn['first_wrong'] is not None:
    raise AssertionError(('synthetic OOD nonoverlap',res_syn))
print('synthetic OOD nonoverlap:', health['synthetic_ood_nonoverlap'])

synthetic OOD nonoverlap: 1200/1200


In [12]:
with open(AUDIT_JSON,'w') as f:
    json.dump(health,f,indent=2)
with open(AUDIT_CSV,'w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=list(health.keys()))
    writer.writeheader(); writer.writerow(health)
for zp in [ZIP_PATH, SUBMISSION_ZIP, GENERIC_ZIP]:
    if zp.exists():
        zp.unlink()
    with zipfile.ZipFile(zp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(ONNX_PATH,arcname=f'{TASK_ID}.onnx')
print('wrote:', ZIP_PATH, SUBMISSION_ZIP, GENERIC_ZIP)
print('zip contents:', zipfile.ZipFile(SUBMISSION_ZIP).namelist())
print('audit:', AUDIT_JSON, AUDIT_CSV)

wrote: /kaggle/working/task092_static_graph_submission.zip /kaggle/working/task092_zero_pad_submission.zip /kaggle/working/submission.zip
zip contents: ['task092.onnx']
audit: /kaggle/working/task092_zero_pad_audit.json /kaggle/working/task092_zero_pad_audit.csv
